In [2]:
import pandas as pd

INPUT_CSV = "fitment_output.csv"
OUTPUT_CSV = "fitment_outliers.csv"

# Removed Application
CHECK_COLUMNS = ["Details", "Notes"]

df = pd.read_csv(INPUT_CSV).fillna("")

outlier_rows = []

for url, group in df.groupby("URL"):
    for col in CHECK_COLUMNS:
        counts = group[col].value_counts(dropna=False)

        # Skip if only one unique value
        if len(counts) <= 1:
            continue

        most_common_count = counts.iloc[0]

        for value, count in counts.items():
            # Flag rare values
            if count == 1 or count <= most_common_count * 0.10:
                matched_rows = group[group[col] == value].copy()

                matched_rows["Outlier Column"] = col
                matched_rows["Outlier Value"] = value
                matched_rows["Outlier Count"] = count
                matched_rows["Most Common Count"] = most_common_count
                matched_rows["Reason"] = f"{col} appears only {count} time(s) for this URL"

                outlier_rows.append(matched_rows)

if outlier_rows:
    outliers = pd.concat(outlier_rows, ignore_index=True)

    outliers = outliers.drop_duplicates(
        subset=["URL", "Application", "Details", "Notes", "Outlier Column"]
    )

    outliers.to_csv(OUTPUT_CSV, index=False)
    print(f"Done. Outliers saved to {OUTPUT_CSV}")
    print(f"Outlier rows found: {len(outliers)}")
else:
    print("No outliers found.")

Done. Outliers saved to fitment_outliers.csv
Outlier rows found: 26


In [3]:
import pandas as pd

INPUT_CSV = "fitment_output.csv"
OUTPUT_CSV = "fitment_url_differences.csv"

df = pd.read_csv(INPUT_CSV).fillna("")

# Normalize for cleaner matching
df["Details_clean"] = df["Details"].str.strip()
df["Notes_clean"] = df["Notes"].str.strip()

urls = df["URL"].dropna().unique()

if len(urls) != 2:
    raise ValueError(f"This script expects exactly 2 URLs. Found {len(urls)} URLs.")

url_1, url_2 = urls[0], urls[1]

df1 = df[df["URL"] == url_1].copy()
df2 = df[df["URL"] == url_2].copy()

set1 = set(zip(df1["Details_clean"], df1["Notes_clean"]))
set2 = set(zip(df2["Details_clean"], df2["Notes_clean"]))

only_in_url_1 = set1 - set2
only_in_url_2 = set2 - set1

results = []

for details, notes in only_in_url_1:
    results.append({
        "Found In": "URL 1 Only",
        "URL": url_1,
        "Details": details,
        "Notes": notes
    })

for details, notes in only_in_url_2:
    results.append({
        "Found In": "URL 2 Only",
        "URL": url_2,
        "Details": details,
        "Notes": notes
    })

out = pd.DataFrame(results)

if out.empty:
    print("No differences found. Details and Notes match between the 2 URLs.")
else:
    out.to_csv(OUTPUT_CSV, index=False)
    print(f"Differences saved to {OUTPUT_CSV}")
    print(f"Different rows found: {len(out)}")

Differences saved to fitment_url_differences.csv
Different rows found: 74


In [4]:
import pandas as pd

INPUT_CSV = "fitment_output.csv"
OUTPUT_CSV = "details_differences.csv"

df = pd.read_csv(INPUT_CSV).fillna("")

# Clean Details for consistent matching
df["Details_clean"] = df["Details"].astype(str).str.strip()

urls = df["URL"].dropna().unique()

if len(urls) != 2:
    raise ValueError(f"This script expects exactly 2 URLs. Found {len(urls)} URLs.")

url_1, url_2 = urls[0], urls[1]

df1 = df[df["URL"] == url_1]
df2 = df[df["URL"] == url_2]

set1 = set(df1["Details_clean"])
set2 = set(df2["Details_clean"])

only_in_url_1 = set1 - set2
only_in_url_2 = set2 - set1

results = []

for details in only_in_url_1:
    results.append({
        "Found In": "URL 1 Only",
        "URL": url_1,
        "Details": details
    })

for details in only_in_url_2:
    results.append({
        "Found In": "URL 2 Only",
        "URL": url_2,
        "Details": details
    })

out = pd.DataFrame(results)

if out.empty:
    print("No differences found. Details match between the 2 URLs.")
else:
    out.to_csv(OUTPUT_CSV, index=False)
    print(f"Differences saved to {OUTPUT_CSV}")
    print(f"Different Details found: {len(out)}")

Differences saved to details_differences.csv
Different Details found: 2
